# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Croissant dataset `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview

Explore available record sets, fields, and their `@id`s in the dataset. All elements are referenced using their `@id` as required by the Croissant schema. This allows reproducible, schema-compliant code and easier mapping of fields to the actual dataset structure.

> **Note:** In Croissant, a *record set* represents a set of records typically corresponding to a table or file, each with fields (columns) defined under unique `@id`s.

In [ ]:
from pprint import pprint

print("Available record sets (by @id):\n")
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # Sometimes .recordSet can be empty, try top-level 'schema:hasPart' for Croissant datasets
    record_sets = getattr(metadata, 'hasPart', [])
record_set_ids = []
for record_set in record_sets:
    recid = getattr(record_set, '@id', None) if hasattr(record_set, '@id') else record_set.get('@id', None)
    print(f"- {recid}")
    record_set_ids.append(recid)

# List fields available in each record set, referenced by their @id.
print("\nFields for each record set:")
for i, record_set in enumerate(record_sets):
    rec_id = getattr(record_set, '@id', None) if hasattr(record_set, '@id') else record_set.get('@id', None)
    fields = getattr(record_set, 'field', []) if hasattr(record_set, 'field') else record_set.get('field', [])
    print(f"Record set [{i}] @id: {rec_id}")
    for field in fields:
        # Each field is a Field object or dict with @id
        fid = getattr(field, '@id', field.get('@id', 'unknown'))
        fname = getattr(field, 'name', field.get('name', ''))
        print(f"  - Field: {fid} | name: {fname}")
    print("")

# For datasets where dataset.metadata.recordSet is empty, you may use dataset.record_sets dict
if not record_set_ids and hasattr(dataset, 'record_sets'):
    print("Record sets found in dataset.record_sets:")
    for k in dataset.record_sets:
        print(f"- {k}")
        record_set_ids.append(k)

## 3. Data Extraction

Load data from a chosen record set into a DataFrame for analysis. Use the record set and field `@id`s as explored above.

For this dataset, if the schema only has a single record set or the main data table, we'll extract it dynamically.

In [ ]:
# If no record sets were detected above, try from dataset.record_sets
record_sets_final = [rid for rid in record_set_ids if rid is not None] or list(getattr(dataset, 'record_sets', {}).keys())
dataframes = {}

# If the dataset has no explicit record sets, fallback to the main record set (usually main data file)
if not record_sets_final:
    raise RuntimeError("No record sets detected! Please check the dataset's Croissant schema.")
# We'll use the first record set as an example
main_record_set_id = record_sets_final[0]

# Extract all records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df

print(f"Loaded columns for record set '@id': {main_record_set_id}\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

This section demonstrates basic EDA: filtering records, normalizing numeric fields, and grouping by categorical fields according to their `@id` reference.

> **Tip:** Examine the output columns above. Replace `<numeric_field_id>` and `<group_field_id>` below with the appropriate column name (the field `@id` as provided in the schema or actual column name if loaded), e.g., `'MW [kDa]'` (Molecular Weight), `'Normalised Abundance'`, `'accession'`, etc.

In [ ]:
# --- Set up field IDs for EDA ---
# Example: use 'MW [kDa]' (Molecular Weight) as a numeric field if present
numeric_field = None
group_field = None
# Find a likely numeric field and group field
for col in df.columns:
    if 'MW' in col or 'Abund' in col or 'peptide' in col:
        numeric_field = numeric_field or col
    if 'accession' in col or 'description' in col or 'Type' in col:
        group_field = group_field or col

assert numeric_field is not None, "No numeric field found in columns!"

print(f"Using numeric_field: '{numeric_field}' and group_field: '{group_field}'\n")
# Try to convert to numeric (if not already)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter for values above a certain threshold
threshold = df[numeric_field].quantile(0.5)  # Median as an example threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field + '_normalized'] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by the categorical field (if it exists)
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the chosen numeric field and relationship with the group field (if present).

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
df[numeric_field].hist(bins=20)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot grouped by group_field (if exists)
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 4))
    df.boxplot(column=numeric_field, by=group_field, rot=90)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- This notebook loaded and explored the Croissant FAIR² dataset via `mlcroissant`. All fields and record sets were referenced by their unique `@id` as per schema best practices.
- Using the provided field and record set IDs, we extracted records, performed EDA, and visualized data distributions.

**Next Steps:** You can extend this notebook to apply advanced processing, machine learning, or integrate other omics datasets using similar Croissant-compliant approaches.